In [0]:

# %pip uninstall -y cudf-cu13 cuml-cu13 cugraph-cu13 cuvs-cu13 libcuml-cu13 libcuvs-cu13 libraft-cu13 pylibcudf-cu13 pylibraft-cu13 rmm-cu13

# Install PyTorch with CUDA 12.9
%pip install --upgrade --no-cache-dir \
  --index-url https://download.pytorch.org/whl/cu129 \
  "torch==2.9.1" "torchvision==0.24.1" "torchaudio==2.9.1"

# Install CUDA-12 RAPIDS packages, pinned to the same RAPIDS release
%pip install --upgrade --no-cache-dir \
  "cupy-cuda12x>=13.6.0" \
  "rmm-cu12==26.02.*" \
  "cudf-cu12==26.02.*" \
  "cuml-cu12==26.02.*"

dbutils.library.restartPython()

In [0]:
import cuml
%load_ext cuml.accel

In [0]:
import cuml.cluster
print("Available clustering algorithms in cuml.cluster:")
print([attr for attr in dir(cuml.cluster) if not attr.startswith('_')])

In [0]:

import time
import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
pio.renderers.default = "browser"

from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, adjusted_rand_score
from multiprocessing import Manager
from cuml.manifold import UMAP
from cuml.cluster import KMeans  # GPU-accelerated KMeans

In [0]:
# -------------------------
# Load cached latent representations from Representation_Extracting
# -------------------------
from pathlib import Path
import numpy as np
import json

# Configuration - MUST match Representation_Extracting notebook
cache_dir = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/cache_dir")
encoder_name = "PFTSleep"
num_files = 2
frequency = 125
win_length = 750
hop_length = 750
max_seq_len_sec = 8 * 3600

# Build cache tag (same function as in Representation_Extracting)
def cache_tag(encoder_name, num_files, frequency, win_length, hop_length, max_seq_len_sec):
    return f"{encoder_name}__files{num_files}__freq{frequency}__win{win_length}__hop{hop_length}__max{max_seq_len_sec}"

tag = cache_tag(encoder_name, num_files, frequency, win_length, hop_length, max_seq_len_sec)

# Load all cached arrays
Z = np.load(cache_dir / f"Z__{tag}.npz")['Z']
night_id = np.load(cache_dir / f"night_id__{tag}.npy")
time_idx = np.load(cache_dir / f"time_idx__{tag}.npy")
zarr_file_idx = np.load(cache_dir / f"zarr_file_idx__{tag}.npy")
windows_start_idx = np.load(cache_dir / f"windows_start_idx__{tag}.npy")
zarr_files_list = np.load(cache_dir / f"zarr_files_list__{tag}.npy", allow_pickle=True).tolist()

# Load zarr ID mapping
with open(cache_dir / f"zarr_files_list__{tag}.json", 'r') as f:
    zarr_files_list = json.load(f)

print(f"✅ Loaded latent representations from Representation_Extracting:")
print(f"  Z shape: {Z.shape}")
print(f"  Night IDs: {len(np.unique(night_id))} unique nights")
print(f"  Zarr files: {len(zarr_files_list)} files")
print(f"  Zarr file indices: {len(np.unique(zarr_file_idx))} unique")
print(f"  Time indices range: {time_idx.min():.1f}s to {time_idx.max():.1f}s")
print(f"\n📁 Zarr files loaded:")
for uid, idx in sorted(zarr_id_map.items(), key=lambda x: x[1]):
    print(f"  [{idx}] {uid}")

In [0]:
# ------------------------------------------------------------
# 1) Normalize RAW 512-D (angular geometry)
# ------------------------------------------------------------

X = normalize(Z.astype(np.float32), norm="l2")
N, D = X.shape

print(f"✅ Normalized latent space ready for clustering:")
print(f"  Shape: {X.shape}")
print(f"  Data type: {X.dtype}")

In [0]:
# ------------------------------------------------------------

# 2) RANDOM HYPERPLANE LSH

# ------------------------------------------------------------

def lsh_hash(X, n_bits, seed=42):

    rng = np.random.default_rng(seed)

    hyperplanes = rng.standard_normal((n_bits, D)).astype(np.float32)

    proj = X @ hyperplanes.T

    bits = (proj > 0).astype(np.uint8)

    powers = (1 << np.arange(n_bits, dtype=np.uint64))

    return (bits.astype(np.uint64) * powers).sum(axis=1)
 
def bucket_stats(hash_vals):

    unique, counts = np.unique(hash_vals, return_counts=True)

    return {

        "nonempty": len(unique),

        "singleton_frac": float(np.sum(counts == 1)) / N,

        "median_size": float(np.median(counts)),

        "max_size": int(np.max(counts)),

    }
 
# ------------------------------------------------------------

# 3) FIND GOOD LSH BIT DEPTH

# ------------------------------------------------------------

bits_range = range(4, 22)

target_buckets = 2000
 
best_bits = None

best_score = None
 
for b in bits_range:

    h = lsh_hash(X, n_bits=b)

    stats = bucket_stats(h)

    score = (

        abs(stats["nonempty"] - target_buckets)

        + 3000 * stats["singleton_frac"]

    )

    print(f"bits={b} | buckets={stats['nonempty']} | singleton_frac={stats['singleton_frac']:.3f}")

    if best_score is None or score < best_score:

        best_score = score

        best_bits = b
 
print("Chosen LSH bits:", best_bits)
 
# ------------------------------------------------------------


In [0]:
# 4) LSH-MEANS INITIALIZATION
# ------------------------------------------------------------

def lsh_means_init(X, n_clusters, n_bits):
    h = lsh_hash(X, n_bits=n_bits)
    unique, inv, counts = np.unique(h, return_inverse=True, return_counts=True)
 
    # Largest buckets first
    order = np.argsort(counts)[::-1]
    keep = order[:5000]
 
    centroids = []
    weights = []
 
    for idx in keep:
        mask = inv == idx
        if mask.sum() == 0:
            continue
        c = X[mask].mean(axis=0)
        centroids.append(c)
        weights.append(mask.sum())
 
    C = np.vstack(centroids)
    weights = np.array(weights)
 
    # cuML KMeans doesn't support sample_weight, so we replicate centroids proportionally
    # Scale weights to reasonable range (max 100 copies per centroid)
    scaled_weights = (weights / weights.max() * 100).astype(int) + 1
    weighted_C = np.vstack([np.tile(C[i], (scaled_weights[i], 1)) for i in range(len(C))])
    
    km = KMeans(
        n_clusters=n_clusters,
        max_iter=300,
        n_init=10,
        random_state=42
    )
 
    km.fit(weighted_C)
 
    return normalize(km.cluster_centers_, norm="l2")


# ------------------------------------------------------------
# 5) SWITCH RATE
# ------------------------------------------------------------

def switch_rate(labels, night_id, time_idx):
    rates = []
    for nid in np.unique(night_id):
        m = night_id == nid
        if m.sum() < 2:
            continue
        t = time_idx[m]
        y = labels[m]
        order = np.argsort(t)
        y = y[order]
        rates.append(np.mean(y[1:] != y[:-1]))
    return float(np.mean(rates))
 
# ------------------------------------------------------------

In [0]:
# 6) SWEEP K ON RAW (using silhouette score only)
# ------------------------------------------------------------
import time

k_values = list(range(4, 25))  # Reduced from 8-18 to 8-15 for faster sweep
results = []
 
rng = np.random.default_rng(42)
sample_n = min(20000, len(X))
sil_idx = rng.choice(len(X), sample_n, replace=False)

X_sil = X[sil_idx] 

print(f"🚀 Starting K-sweep with GPU KMeans (testing {len(k_values)} k-values)...")
print(f"   Data: {X.shape[0]:,} windows, Silhouette sample: {sample_n:,} windows\n")

for i, k in enumerate(k_values, 1):
    start_time = time.time()
    print(f"[{i}/{len(k_values)}] Testing k={k}...", end=" ", flush=True)
    
    # LSH-means initialization
    init_centers = lsh_means_init(X, k, best_bits)

    # GPU KMeans clustering
    km = KMeans(
        n_clusters=k,
        init=init_centers,
        max_iter=30,
        n_init=1,
        random_state=42
    )

    labels = km.fit_predict(X)
 
    # Calculate silhouette score (only metric used for best_k)
    sil = silhouette_score(X_sil, labels[sil_idx])
    
    elapsed = time.time() - start_time
    results.append((k, sil))
    print(f"silhouette={sil:.4f} ({elapsed:.1f}s)")
 
# Determine best k based on silhouette score only
best_k = max(results, key=lambda x: x[1])[0]
best_sil = max(results, key=lambda x: x[1])[1]

print(f"\n✅ Best k: {best_k} (silhouette score: {best_sil:.4f})")
print(f"\n📊 All results:")
for k, sil in results:
    marker = " ← BEST" if k == best_k else ""
    print(f"   k={k:2d}: silhouette={sil:.4f}{marker}")
 
# ------------------------------------------------------------

In [0]:
# 7) FINAL FIT
# ------------------------------------------------------------

final_init = lsh_means_init(X, best_k, best_bits)
 
final_km = KMeans(
    n_clusters=best_k,
    init=final_init,
    max_iter=300,
    n_init=1,
    random_state=42
)
 
cluster_id = final_km.fit_predict(X).astype(int)
print("Final cluster ids:", np.unique(cluster_id))
 
# ------------------------------------------------------------

In [0]:
# 8) UMAP ON RAW 512-D (NO PCA)
# ------------------------------------------------------------

reducer = UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=3,
    metric="euclidean",
    random_state=42,
    verbose=True
)
 
embedding = reducer.fit_transform(X)
 
fig = go.Figure(
    data=go.Scatter3d(
        x=embedding[:,0],
        y=embedding[:,1],
        z=embedding[:,2],
        mode="markers",
        marker=dict(
            size=3,
            color=cluster_id,
            showscale=True
        )
    )
)
 
fig.update_layout(
    title=f"RAW 512-D UMAP | k={best_k} | bits={best_bits}",
    height=800
)
 
display(fig)

In [0]:
# -------------------------
# Save cluster assignments with zarr file mapping
# -------------------------
import pandas as pd

# Create comprehensive mapping dataframe
cluster_mapping = pd.DataFrame({
    'window_idx': np.arange(len(cluster_id)),
    'cluster_id': cluster_id,
    'zarr_file_idx': zarr_file_idx,
    'night_id': night_id,
    'time_idx_sec': time_idx,
    'windows_start_idx': windows_start_idx
})

# Add zarr file metadata
idx_to_uid = {v: k for k, v in zarr_id_map.items()}
cluster_mapping['zarr_uid'] = cluster_mapping['zarr_file_idx'].map(idx_to_uid)
cluster_mapping['zarr_file_path'] = cluster_mapping['zarr_file_idx'].apply(
    lambda x: zarr_files_list[x] if x < len(zarr_files_list) else None
)

# Save to cache directory
output_path = cache_dir / f"cluster_assignments__{tag}.csv"
cluster_mapping.to_csv(output_path, index=False)

print(f"✅ Saved cluster assignments to: {output_path}")
print(f"\n📊 Cluster distribution:")
print(cluster_mapping['cluster_id'].value_counts().sort_index())
print(f"\n📁 Data points per zarr file:")
zarr_summary = cluster_mapping.groupby('zarr_uid').agg({
    'window_idx': 'count',
    'cluster_id': lambda x: x.nunique()
}).rename(columns={'window_idx': 'num_windows', 'cluster_id': 'num_clusters'})
print(zarr_summary)

# Display sample
print(f"\n🔍 Sample cluster assignments:")
display(cluster_mapping.head(20))

In [0]:
# -------------------------
# Restructure demographics CSV to match zarr folder order
# -------------------------
import pandas as pd
from pathlib import Path

# Path to zarrs folder
zarrs_dir = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/zarrs")

# Get all zarr files in order
zarr_files = sorted(zarrs_dir.glob("*.zarr"))
print(f"📁 Found {len(zarr_files)} zarr files")

# Extract nsrrid from each zarr filename (e.g., "shhs1-200002.zarr" -> 200002)
zarr_nsrrids = []
for zarr_file in zarr_files:
    # Extract the number after "shhs1-"
    nsrrid = int(zarr_file.stem.split('-')[-1])
    zarr_nsrrids.append(nsrrid)
    print(f"  {zarr_file.name} -> nsrrid: {nsrrid}")

# Load the original demographics CSV
demographics_df = pd.read_csv('/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel.csv')
print(f"\n📊 Original demographics shape: {demographics_df.shape}")

# Create a mapping dataframe with the desired order
order_df = pd.DataFrame({'nsrrid': zarr_nsrrids, 'order': range(len(zarr_nsrrids))})

# Merge with demographics to get the order column
demographics_ordered = demographics_df.merge(order_df, on='nsrrid', how='inner')

# Sort by the order column
demographics_ordered = demographics_ordered.sort_values('order')

# Drop the order column
demographics_ordered = demographics_ordered.drop('order', axis=1)

print(f"\n✅ Restructured demographics shape: {demographics_ordered.shape}")
print(f"📋 Matched {len(demographics_ordered)} / {len(zarr_nsrrids)} zarr files")

# Show the new order
print(f"\n🔍 First 10 nsrrids in new order:")
print(demographics_ordered['nsrrid'].head(10).tolist())

# Save the restructured CSV
output_path = '/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel_ordered.csv'
demographics_ordered.to_csv(output_path, index=False)

print(f"\n💾 Saved restructured CSV to:")
print(f"   {output_path}")

# Verify the order matches
print(f"\n✅ Verification:")
print(f"   Zarr order: {zarr_nsrrids[:5]}")
print(f"   CSV order:  {demographics_ordered['nsrrid'].head(5).tolist()}")
print(f"   Match: {zarr_nsrrids[:5] == demographics_ordered['nsrrid'].head(5).tolist()}")

In [0]:
# -------------------------
# Load demographics from sleep_excel and join with cluster assignments
# -------------------------
import pandas as pd

# Load the ordered demographics CSV
demographics_df = pd.read_csv('/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel_ordered.csv')

print(f"✅ Loaded demographics: {demographics_df.shape}")
print(f"\n📝 First few nsrrid values:")
print(demographics_df['nsrrid'].head(10))
print(f"\nnsrrid dtype: {demographics_df['nsrrid'].dtype}")

# Extract nsrrid from zarr_uid (e.g., "shhs1-200002" -> 200002)
# Use \d+$ to get all digits at the end of the string
cluster_mapping['nsrrid'] = cluster_mapping['zarr_uid'].str.extract(r'(\d+)$')[0].astype(int)

print(f"\n🔍 Extracted nsrrid from zarr_uid:")
print(cluster_mapping[['zarr_uid', 'nsrrid']].drop_duplicates())

# Ensure demographics nsrrid is int
demographics_df['nsrrid'] = demographics_df['nsrrid'].astype(int)

# Join cluster assignments with demographics
cluster_demo = cluster_mapping.merge(
    demographics_df[['nsrrid', 'age_s1', 'gender', 'race_s1', 'ethnicity_s1', 'smokecat_s1', 'Diabetes', 'COPD', 'bmi_s1']],
    on='nsrrid',
    how='left'
)

print(f"\n📊 Joined data shape: {cluster_demo.shape}")
print(f"✅ Successfully matched {cluster_demo['age_s1'].notna().sum()} / {len(cluster_demo)} records")

# Display sample
display(cluster_demo.head(20))

In [0]:
# -------------------------
# Analyze demographics by cluster
# -------------------------

# Define age ranges
def categorize_age(age):
    if pd.isna(age):
        return 'Unknown'
    elif age <= 18:
        return '0-18'
    elif age <= 38:
        return '19-38'
    elif age <= 57:
        return '39-57'
    elif age <= 76:
        return '58-76'
    elif age <= 95:
        return '77-95'
    else:
        return '96-114'

cluster_demo['age_range'] = cluster_demo['age_s1'].apply(categorize_age)

# Define BMI categories
def categorize_bmi(bmi):
    if pd.isna(bmi):
        return 'Unknown'
    elif bmi < 18.5:
        return 'Underweight (<18.5)'
    elif bmi < 25:
        return 'Healthy Weight (18.5-24.9)'
    elif bmi < 30:
        return 'Overweight (25-29.9)'
    elif bmi < 35:
        return 'Obesity Class I (30-34.9)'
    elif bmi < 40:
        return 'Obesity Class II (35-39.9)'
    else:
        return 'Obesity Class III (>=40)'

cluster_demo['bmi_category'] = cluster_demo['bmi_s1'].apply(categorize_bmi)

# Function to calculate percentages by cluster
def cluster_demographics(df, column, cluster_col='cluster_id'):
    """Calculate percentage distribution of a column within each cluster"""
    result = df.groupby([cluster_col, column]).size().unstack(fill_value=0)
    result_pct = result.div(result.sum(axis=1), axis=0) * 100
    return result_pct.round(2)

print("=" * 80)
print("DEMOGRAPHIC ANALYSIS BY CLUSTER")
print("=" * 80)

# Age Range Distribution
print("\n📊 AGE RANGE DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
age_dist = cluster_demographics(cluster_demo, 'age_range')
display(age_dist)

# Gender Distribution
print("\n👥 GENDER DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
gender_dist = cluster_demographics(cluster_demo, 'gender')
display(gender_dist)

# Race Distribution
print("\n🌍 RACE DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
race_dist = cluster_demographics(cluster_demo, 'race_s1')
display(race_dist)

# Ethnicity Distribution
print("\n🗺️ ETHNICITY DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
ethnicity_dist = cluster_demographics(cluster_demo, 'ethnicity_s1')
display(ethnicity_dist)

# Smoking Status Distribution
print("\n🚬 SMOKING STATUS DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
smoke_dist = cluster_demographics(cluster_demo, 'smokecat_s1')
display(smoke_dist)

# Diabetes Distribution
print("\n💉 DIABETES DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
diabetes_dist = cluster_demographics(cluster_demo, 'Diabetes')
display(diabetes_dist)

# COPD Distribution
print("\n🫁 COPD DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
copd_dist = cluster_demographics(cluster_demo, 'COPD')
display(copd_dist)

# BMI Category Distribution
print("\n⚖️ BMI CATEGORY DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
bmi_dist = cluster_demographics(cluster_demo, 'bmi_category')
display(bmi_dist)

# Summary statistics
print("\n📈 CLUSTER SUMMARY STATISTICS")
print("-" * 80)
summary = cluster_demo.groupby('cluster_id').agg({
    'nsrrid': 'count',
    'age_s1': ['mean', 'std', 'min', 'max'],
    'gender': [
        lambda x: (x == 'male').sum() / len(x) * 100,
        lambda x: (x == 'female').sum() / len(x) * 100
    ],
    'Diabetes': lambda x: (x == 'Yes').sum() / x.notna().sum() * 100 if x.notna().sum() > 0 else 0,
    'COPD': lambda x: (x == 'Yes').sum() / x.notna().sum() * 100 if x.notna().sum() > 0 else 0
}).round(2)
summary.columns = ['N', 'Age_Mean', 'Age_SD', 'Age_Min', 'Age_Max', 'Male_%', 'Female_%', 'Diabetes_%', 'COPD_%']
display(summary)

In [0]:
# -------------------------
# Enhanced 3D UMAP visualization with demographic hover info
# -------------------------
import plotly.graph_objects as go

# Aggregate cluster_demo to one row per window (take first occurrence per window_idx)
cluster_demo_unique = cluster_demo.drop_duplicates(subset='window_idx')

# Create hover text with all demographic info
hover_text = []
for idx, row in cluster_demo_unique.iterrows():
    text = (
        f"<b>nsrrid:</b> {row['nsrrid']}<br>"
        f"<b>Zarr File:</b> {row['zarr_uid']}<br>"
        f"<b>Cluster:</b> {row['cluster_id']}<br>"
        f"<b>Window:</b> {row['window_idx']}<br>"
        f"<b>Time:</b> {row['time_idx_sec']:.1f}s<br>"
        f"<br><b>Demographics:</b><br>"
        f"Age: {row['age_s1']} ({row['age_range']})<br>"
        f"Gender: {row['gender']}<br>"
        f"Race: {row['race_s1']}<br>"
        f"Ethnicity: {row['ethnicity_s1']}<br>"
        f"Smoking: {row['smokecat_s1']}<br>"
        f"Diabetes: {row['Diabetes']}<br>"
        f"COPD: {row['COPD']}"
    )
    hover_text.append(text)

# Create 3D scatter plot
fig = go.Figure(data=go.Scatter3d(
    x=embedding[:, 0],
    y=embedding[:, 1],
    z=embedding[:, 2],
    mode='markers',
    marker=dict(
        size=3,
        color=cluster_demo_unique['cluster_id'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Cluster ID"),
        line=dict(width=0)
    ),
    text=hover_text,
    hovertemplate='%{text}<extra></extra>',
    name='Data Points'
))

fig.update_layout(
    title=f"3D UMAP Visualization with Demographics | k={best_k} clusters",
    scene=dict(
        xaxis_title='UMAP 1',
        yaxis_title='UMAP 2',
        zaxis_title='UMAP 3',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.5)
        )
    ),
    height=800,
    width=1200,
    hovermode='closest'
)

display(fig)

print(f"\n✅ Interactive 3D UMAP created with {len(cluster_demo_unique)} points")
print(f"   Hover over points to see nsrrid and demographic information")

In [0]:
# -------------------------
# Save cluster assignments with demographics
# -------------------------

# Save the complete dataset
output_path_demo = cache_dir / f"cluster_assignments_with_demographics__{tag}.csv"
cluster_demo.to_csv(output_path_demo, index=False)

print(f"✅ Saved enhanced cluster assignments to:")
print(f"   {output_path_demo}")
print(f"\n📊 File contains {len(cluster_demo)} rows with:")
print(f"   - Cluster assignments")
print(f"   - Zarr file mapping")
print(f"   - Complete demographics (age, gender, race, ethnicity, smoking, diabetes, COPD)")

# GPU-Accelerated Clustering Complete ✅

## Workflow Summary

This notebook performs GPU-accelerated clustering on latent representations extracted by the **Representation_Extracting** notebook.

### Pipeline Steps

1. **Load Cached Latents** (Cell 5)
   - Loads `Z`, `night_id`, `time_idx`, `zarr_file_idx`, `windows_start_idx`, `zarr_files_list`
   - Loads `zarr_id_map.json` for file mapping

2. **Normalize** (Cell 7)
   - L2 normalization for angular geometry

3. **LSH Hashing** (Cell 8)
   - Random hyperplane LSH to find optimal bit depth
   - Targets ~2000 buckets for initialization

4. **LSH-Means Initialization** (Cell 9)
   - Uses LSH buckets to create smart initial centroids
   - **GPU-accelerated KMeans** for centroid refinement

5. **K-Value Sweep** (Cell 10)
   - Tests k values from 8-17
   - **GPU-accelerated KMeans** for each k
   - Evaluates using silhouette score

6. **Final Clustering** (Cell 11)
   - **GPU-accelerated KMeans** with best k
   - Assigns cluster IDs to all windows

7. **UMAP Visualization** (Cell 12)
   - 3D UMAP embedding colored by cluster

8. **Save Results** (Cell 13)
   - Creates comprehensive CSV with cluster assignments
   - Maps back to source zarr files
   - Includes all metadata for further analysis

## Output Files

**Saved to**: `cache_dir / cluster_assignments__{tag}.csv`

**Columns**:
- `window_idx`: Sequential window index
- `cluster_id`: Assigned cluster ID (from GPU KMeans)
- `zarr_file_idx`: Numeric zarr file ID
- `night_id`: Night/batch identifier
- `time_idx_sec`: Time index in seconds
- `windows_start_idx`: Start index in original zarr
- `zarr_uid`: Original zarr file UID
- `zarr_file_path`: Full path to source zarr file

## GPU Acceleration

✅ **All clustering operations use cuML GPU KMeans**
- Faster than CPU sklearn by 10-100x on large datasets
- Runs on T4 GPU (Standard_NC4as_T4_v3)
- Compatible with LSH-means initialization strategy

## Next Steps

Use the `cluster_assignments__{tag}.csv` file to:
- Analyze cluster characteristics
- Extract representative windows from each cluster
- Map clusters back to original zarr files for detailed inspection
- Perform downstream analysis on specific clusters